# Class 5 Chatbot — Manual Testing Notebook
Run cells top-to-bottom to inspect and test the backend pipeline.

In [ ]:
# Cell 1: Imports and path setup
import sys, os
sys.path.insert(0, os.path.abspath('.'))

from backend.config import settings
print('Project root:', settings.project_root)
print('Vectorstore :', settings.vectorstore_path)
print('Collection  :', settings.chroma_collection_name)
print('Ollama URL  :', settings.ollama_base_url)
print('Model       :', settings.ollama_model)

In [ ]:
# Cell 2: Discover available PDFs
from backend.services.catalog import build_evs_subject, get_all_subjects

evs = build_evs_subject()
print(f'EVS chapters ({len(evs.chapters)} total):')
for ch in evs.chapters:
    status = '✓ ready' if (ch.available and ch.indexed) else ('PDF found, not indexed' if ch.available else 'no PDF')
    print(f'  {ch.chapter_id}  {ch.title[:45]:<45}  {status}')

In [ ]:
# Cell 3: Inspect PDF text extraction for one chapter
from backend.services.indexing import extract_pages
from backend.services.catalog import get_evs_chapter_pdf_path

pdf_path = get_evs_chapter_pdf_path('EVS_CH01')
print('PDF:', pdf_path)

pages = extract_pages(pdf_path)
print(f'Extracted {len(pages)} pages with text')
for p in pages:
    print(f'  Page {p["page"]}: {len(p["text"])} chars')

# Show the first 400 chars of page 1
if pages:
    print('\n--- Page 1 excerpt ---')
    print(pages[0]['text'][:400])

In [ ]:
# Cell 4: Text cleaning and chunking
from backend.services.indexing import clean_text, chunk_text
from backend.config import settings

if pages:
    sample_text = pages[0]['text']
    chunks = chunk_text(sample_text, settings.chunk_size, settings.chunk_overlap)
    print(f'Page 1 -> {len(chunks)} chunks')
    print(f'Chunk 1 ({len(chunks[0])} chars):')
    print(chunks[0][:300])

In [ ]:
# Cell 5: Load embedding model
from backend.services.indexing import get_embedding_model
model = get_embedding_model()
print('Model:', type(model).__name__)
# Test embedding dimensions
import numpy as np
test_vec = model.encode('passage: What is history?', normalize_embeddings=True)
print('Embedding shape:', test_vec.shape)
print('First 5 values:', test_vec[:5].round(4))

In [ ]:
# Cell 6: Inspect ChromaDB collection
from backend.services.indexing import get_chroma_collection, get_indexed_chapter_ids
import chromadb

col = get_chroma_collection()
total = col.count()
indexed_ids = get_indexed_chapter_ids(col)
print(f'Collection: {col.name}')
print(f'Total chunks: {total}')
print(f'Indexed chapter IDs ({len(indexed_ids)}): {sorted(indexed_ids)}')

# Peek at metadata
if total > 0:
    peek = col.peek(limit=3)
    print('\nSample metadatas:')
    for m in peek['metadatas']:
        print(' ', m)

In [ ]:
# Cell 7: Run indexing pipeline (skip already-indexed chapters)
# To force re-index: run_indexing(force=True)
from backend.services.indexing import run_indexing
results = run_indexing(force=False)

In [ ]:
# Cell 8: Chapter-specific retrieval
from backend.services.rag import retrieve_passages

chapter_id = 'EVS_CH01'
query = 'What is history and why do we study it?'

passages = retrieve_passages(chapter_id, query, top_k=3)
print(f'Retrieved {len(passages)} passages for {chapter_id}:')
for i, p in enumerate(passages, 1):
    print(f'\n[{i}] Source: {p["source"]} | Page: {p["page_number"]} | Distance: {p["distance"]:.4f}')
    print(p['text'][:300])

In [ ]:
# Cell 9: Verify retrieval isolation between chapters
print('Retrieval isolation test:')
for ch_id in ['EVS_CH01', 'EVS_CH05', 'EVS_CH10']:
    passages = retrieve_passages(ch_id, 'what happened', top_k=5)
    wrong = [p for p in passages if p.get('source','') and ch_id not in p.get('chapter_name','')]
    # check via ChromaDB metadata directly
    col = get_chroma_collection()
    from backend.services.rag import embed_query
    raw = col.query(
        query_embeddings=[embed_query('what happened')],
        n_results=5,
        where={'chapter_id': ch_id},
        include=['metadatas'],
    )
    ids_returned = {m['chapter_id'] for m in raw['metadatas'][0]}
    print(f'  {ch_id}: returned chapter_ids = {ids_returned} | isolated = {ids_returned == {ch_id}}')

In [ ]:
# Cell 10: Ollama health check
import requests
try:
    r = requests.get(f'{settings.ollama_base_url}/api/tags', timeout=5)
    data = r.json()
    models = [m['name'] for m in data.get('models', [])]
    print('Ollama running. Available models:', models)
    model_ok = any(m.startswith(settings.ollama_model) for m in models)
    print(f'{settings.ollama_model} available: {model_ok}')
except Exception as e:
    print(f'Ollama not reachable: {e}')
    print('Start with: ollama serve')

In [ ]:
# Cell 11: Generate an answer with retrieved passages and structured chat
from backend.services.rag import (
    retrieve_passages, build_chat_messages, generate_with_ollama_chat,
    filter_supporting_sources, _remove_thinking_tags
)

chapter_id = 'EVS_CH01'
question = 'What is history? Explain in simple words.'

passages = retrieve_passages(chapter_id, question, top_k=5)
print(f'Retrieved {len(passages)} passages')

messages = build_chat_messages(
    mode='chapter_question',
    query=question,
    passages=passages,
    history=[],
    chapter_name='What is History?',
    flags={},
)
print('Messages count:', len(messages))

print('\nGenerating answer...')
answer = generate_with_ollama_chat(messages, temperature=0.25)
print('\n--- Answer ---')
print(answer)

sources = filter_supporting_sources(passages, 'chapter_question', {})
print('\n--- Sources ---')
for s in sources:
    print(f'  {s.pdf_filename} | page {s.page_number}')


In [ ]:
# Cell 12: Test thinking-tag removal
raw_with_think = '<think>Let me reason about this...</think>\n\nHistory is the study of past events.'
clean = _remove_thinking_tags(raw_with_think)
print('Input :', repr(raw_with_think))
print('Output:', repr(clean))
assert '<think>' not in clean

In [ ]:
# Cell 13: Source citation verification & deduplication
print('Source citation check:')
from backend.services.rag import filter_supporting_sources
fake_passages = [
    {'chapter_name': 'What is History?', 'source': 'Chapter 1.pdf', 'page_number': 2, 'text': 'a', 'distance': 0.20},
    {'chapter_name': 'What is History?', 'source': 'Chapter 1.pdf', 'page_number': 2, 'text': 'b', 'distance': 0.22},
    {'chapter_name': 'What is History?', 'source': 'Chapter 1.pdf', 'page_number': 3, 'text': 'c', 'distance': 0.25},
    {'chapter_name': 'What is History?', 'source': 'Chapter 1.pdf', 'page_number': 4, 'text': 'd', 'distance': 0.45},
]
sources = filter_supporting_sources(fake_passages, 'chapter_question', {})
print(f'4 passages -> {len(sources)} deduplicated, distance-filtered sources')
for s in sources:
    print(f'  {s.pdf_filename} p.{s.page_number}')


In [ ]:
# Cell 14: Full process_chat function test
from backend.services.rag import process_chat
from backend.schemas import ChatRequest

req = ChatRequest(
    chapter_id='EVS_CH01',
    message='What is history?',
    history=[],
    mode='study',
)
resp = process_chat(req)
print('Mode      :', resp.mode)
print('Retrieved :', resp.retrieved_count, 'passages')
print('Sources   :', len(resp.sources))
print('Answer    :', resp.answer[:400])

In [ ]:
# Cell 15: General conversation (no textbook retrieval)
req_general = ChatRequest(
    chapter_id='EVS_CH01',
    message='Hello! How are you?',
    history=[],
    mode='study',  # intent router should detect this as general
)
resp_g = process_chat(req_general)
print('Mode (expected general):', resp_g.mode)
print('Answer:', resp_g.answer[:300])

In [ ]:
# Cell 16: Interactive Q&A loop (run manually, stop with empty input)
chapter_id = 'EVS_CH01'
from backend.schemas import ChatRequest, HistoryMessage
from backend.services.rag import process_chat

history = []
print(f'Chatting about chapter: {chapter_id}')
print('Enter a question (empty to stop):')

while True:
    q = input('You: ').strip()
    if not q:
        break
    req = ChatRequest(chapter_id=chapter_id, message=q, history=history, mode='study')
    resp = process_chat(req)
    print(f'\nAI ({resp.mode}): {resp.answer}')
    if resp.sources:
        for s in resp.sources:
            print(f'  [{s.pdf_filename} p.{s.page_number}]')
    print()
    history.append(HistoryMessage(role='user', content=q))
    history.append(HistoryMessage(role='assistant', content=resp.answer))

In [ ]:
# Cell 17: Test 4-Way Intent Routing Behavior
from backend.services.rag import route_intent
from backend.schemas import HistoryMessage

test_queries = [
    ("Hii", [], "general_chat"),
    ("Tell me a joke", [], "general_chat"),
    ("What is history?", [], "chapter_question"),
    ("What is the difference between history and science?", [], "chapter_plus_general"),
    ("Explain black holes", [], "general_knowledge"),
]

print("Intent Routing Evaluation on EVS Chapter 1:")
for q, hist, expected in test_queries:
    route, passages, _, _ = route_intent(q, hist, "EVS_CH01", "What is History?")
    status = "✓ PASS" if route == expected else f"✗ FAIL ({route})"
    print(f"  {status} | Query: {q:<52} -> {route} (passages: {len(passages)})")


In [ ]:
# Cell 18: Follow-up List Resolution Test ('Explain the second one')
from backend.services.rag import process_chat
from backend.schemas import ChatRequest, HistoryMessage

t1_q = 'What are the three types of sources of history?'
t1_ans = 'The three types of sources of history are: 1. Material sources, 2. Written sources, 3. Oral sources.'
history = [
    HistoryMessage(role='user', content=t1_q),
    HistoryMessage(role='assistant', content=t1_ans),
]
req = ChatRequest(chapter_id='EVS_CH01', message='Explain the second one', history=history, mode='study')
resp = process_chat(req)
print('Mode    :', resp.mode)
print('Sources :', [(s.pdf_filename, s.page_number) for s in resp.sources])
print('Answer  :', resp.answer[:300])

In [ ]:
# Cell 19: Chapter 3 Chronology Verification
from backend.services.rag import process_chat
from backend.schemas import ChatRequest

q_chrono = '''Arrange the events given below in chronological order:
(a) Water bodies appeared on the earth\'s surface.
(b) The sun and the planets revolving around it were created.
(c) Protozoa appeared in water.
(d) An enormous cloud of hot gases and dust was formed in space.'''

req_ch3 = ChatRequest(chapter_id='EVS_CH03', message=q_chrono, history=[], mode='study')
resp_ch3 = process_chat(req_ch3)
print('Chapter 3 Chronology Answer:\n', resp_ch3.answer)
print('\nSources:', [(s.pdf_filename, s.page_number) for s in resp_ch3.sources])

In [ ]:
# Cell 20: Inspect Pipeline Diagnostics (get_diagnostics)
from backend.services.rag import get_diagnostics
import json

diag = get_diagnostics('EVS_CH01', 'What is history?')
print(json.dumps(diag, indent=2))

In [ ]:
# Cell 21: Contextual Follow-up (10 Bullet Points)
from backend.services.rag import process_chat
from backend.schemas import ChatRequest, HistoryMessage

turn1 = process_chat(ChatRequest(chapter_id="EVS_CH01", message="What is history?", history=[]))
h = [
    HistoryMessage(role="user", content="What is history?"),
    HistoryMessage(role="assistant", content=turn1.answer),
]
resp = process_chat(ChatRequest(chapter_id="EVS_CH01", message="Give me bullet points, at least 10.", history=h))
print("Mode    :", resp.mode)
print("Sources :", len(resp.sources))
print("Answer  :\n", resp.answer)


In [ ]:
# Cell 22: General Knowledge Detail Expansion (Black holes -> Detailed)
from backend.services.rag import process_chat
from backend.schemas import ChatRequest, HistoryMessage

r1 = process_chat(ChatRequest(chapter_id="EVS_CH01", message="Explain black holes.", history=[]))
h = [
    HistoryMessage(role="user", content="Explain black holes."),
    HistoryMessage(role="assistant", content=r1.answer),
]
r2 = process_chat(ChatRequest(chapter_id="EVS_CH01", message="Give me a detailed answer.", history=h))
print("Mode    :", r2.mode)
print("Sources :", len(r2.sources))
print("Answer  :\n", r2.answer[:400], "...")


In [ ]:
# Cell 23: Continue a List without Repetition (Freedom fighters -> Add more)
from backend.services.rag import process_chat
from backend.schemas import ChatRequest, HistoryMessage

r1 = process_chat(ChatRequest(chapter_id="EVS_CH01", message="List famous Indian freedom fighters.", history=[]))
print("Part 1:\n", r1.answer)
h = [
    HistoryMessage(role="user", content="List famous Indian freedom fighters."),
    HistoryMessage(role="assistant", content=r1.answer),
]
r2 = process_chat(ChatRequest(chapter_id="EVS_CH01", message="Add more.", history=h))
print("\nPart 2 (Add more):\n", r2.answer)


In [ ]:
# Cell 24: Textbook Coverage Absence Check (Social workers -> 0 citations)
from backend.services.rag import process_chat
from backend.schemas import ChatRequest

resp = process_chat(ChatRequest(chapter_id="EVS_CH01", message="Which social workers are mentioned in this chapter?", history=[]))
print("Mode    :", resp.mode)
print("Sources :", len(resp.sources))
print("Answer  :\n", resp.answer)


In [ ]:
# Cell 25: Educational Roleplay (Shivaji Maharaj -> First Person)
from backend.services.rag import process_chat
from backend.schemas import ChatRequest

resp = process_chat(ChatRequest(chapter_id="EVS_CH01", message="Act like Shivaji Maharaj and tell me your thoughts about Swaraj.", history=[]))
print("Mode    :", resp.mode)
print("Sources :", len(resp.sources))
print("Answer  :\n", resp.answer)


In [ ]:
# Cell 26: Chapter 9 Overview & Anti-Poisoning Check
from backend.services.rag import process_chat
from backend.schemas import ChatRequest, HistoryMessage

print('1. Chapter 9 Overview:')
r_ov = process_chat(ChatRequest(chapter_id='EVS_CH09', message='What are the 3 main topics taught in this chapter?'))
print('Mode   :', r_ov.mode)
print('Sources:', [(s.pdf_filename, s.page_number) for s in r_ov.sources])
print('Answer :\n', r_ov.answer[:300], '...\n')

print('2. Roleplay after Smartphone question (Anti-Poisoning):')
h = [
    HistoryMessage(role='user', content='What does this chapter say about how smartphones and the internet were invented?'),
    HistoryMessage(role='assistant', content='Settled Life and Urban Civilization does not cover smartphones or the internet.'),
]
r_rp = process_chat(ChatRequest(chapter_id='EVS_CH09', message='Act like a scientist or researcher studying the topic of this chapter. Tell me what you do every day.', history=h))
print('Mode   :', r_rp.mode)
print('Sources:', [(s.pdf_filename, s.page_number) for s in r_rp.sources])
print('Answer :\n', r_rp.answer[:300], '...')


In [ ]:
# Cell 27: General Knowledge Factual Accuracy (Rivers in India)
from backend.services.rag import process_chat
from backend.schemas import ChatRequest

r_riv = process_chat(ChatRequest(chapter_id='EVS_CH09', message='Name 4 famous rivers in India.'))
print('Mode   :', r_riv.mode)
print('Sources:', len(r_riv.sources), '(expected 0 fake citations)')
print('Answer :\n', r_riv.answer)
